In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import numpy as np
import re
import time
import shutil
import warnings
from openpyxl.styles import Alignment, Font, Border, Side, PatternFill
from openpyxl.chart import BarChart, LineChart, Reference
import win32com.client as win32
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

USER_ID = "YOUR_USERNAME"
PASS_KEY = "YOUR_PASSWORD"
BASE_DIR = r"C:\Project\Data"
OUTPUT_PATH = os.path.join(BASE_DIR, "final_report.xlsx")
TARGET_URL = "https://portal.example.com/login"
DOWNLOAD_DIR = os.path.join(os.getcwd(), "temp_downloads")

def interaction_handler(wait, xpath, value, label="element"):
    try:
        el = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
        el.click()
        el.clear()
        el.send_keys(value)
        return el
    except Exception as e:
        raise Exception(f"Error on {label}: {e}")

def select_all_visible(driver):
    driver.execute_script("""
        let btns = document.querySelectorAll('button.select-all-class');
        for(let b of btns) {
            if(b.offsetWidth > 0 && b.offsetHeight > 0) {
                b.click();
                break;
            }
        }
    """)

start_date = input("Start Date (YYYY-MM-DD): ")
end_date = input("End Date (YYYY-MM-DD): ")

if os.path.exists(DOWNLOAD_DIR): shutil.rmtree(DOWNLOAD_DIR)
os.makedirs(DOWNLOAD_DIR)

chrome_options = Options()
chrome_options.add_experimental_option("prefs", {"download.default_directory": DOWNLOAD_DIR})
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 120)  

try:
    driver.get(TARGET_URL)
    
    u_field = wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@type='email']")))
    u_field.send_keys(USER_ID + Keys.RETURN)

    p_field = wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@type='password']")))
    time.sleep(1) 
    p_field.send_keys(PASS_KEY + Keys.RETURN)
    
    try:
        wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@type='submit']"))).click()
    except:
        pass

    driver.get("https://portal.example.com/reports")
    time.sleep(5)
    
    interaction_handler(wait, "//input[@formcontrolname='user']", USER_ID, "Portal User")
    try:
        driver.execute_script("arguments[0].click();", wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@value='Login']"))))
    except:
        driver.switch_to.active_element.send_keys(Keys.ENTER)

    time.sleep(5)
    
    menu_el = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Reports Menu']")))
    driver.execute_script("arguments[0].click();", menu_el)
    
    link_el = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@href, 'export')]")))
    driver.execute_script("arguments[0].click();", link_el)
    time.sleep(5)

    drop_1 = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@data-id='filter_1']")))
    driver.execute_script("arguments[0].click();", drop_1)
    select_all_visible(driver)
    driver.execute_script("arguments[0].click();", drop_1)
    
    driver.execute_script(f"document.getElementById('dateStart').value = '{start_date}';")
    driver.execute_script(f"document.getElementById('dateEnd').value = '{end_date}';")
    time.sleep(2)

    drop_2 = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Select Status')]")))
    driver.execute_script("arguments[0].click();", drop_2)
    select_all_visible(driver)
    driver.execute_script("arguments[0].click();", drop_2)
    
    driver.execute_script("arguments[0].click();", wait.until(EC.element_to_be_clickable((By.XPATH, "//button[span[text()='Generate']]"))))
    time.sleep(2)
    driver.execute_script("arguments[0].click();", wait.until(EC.element_to_be_clickable((By.XPATH, "//button[span[text()='Download XLS']]"))))
    
    downloaded_file = None
    for _ in range(600):
        files = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.xlsx')]
        if files:
            p = os.path.join(DOWNLOAD_DIR, files[0])
            if os.path.getsize(p) > 0:
                downloaded_file = p
                break
        time.sleep(1)

    if downloaded_file:
        if os.path.exists(OUTPUT_PATH): os.remove(OUTPUT_PATH)
        shutil.move(downloaded_file, OUTPUT_PATH)
        
        excel_app = win32.Dispatch("Excel.Application")
        wb = excel_app.Workbooks.Open(OUTPUT_PATH)
        wb.Save()
        wb.Close()
        excel_app.Quit()

finally:
    driver.quit()

path_master = os.path.join(BASE_DIR, "master_data.xlsx")
path_raw = OUTPUT_PATH
path_log = os.path.join(BASE_DIR, "historical_log.xlsx")

df_1 = pd.read_excel(path_master)
df_2 = pd.read_excel(path_raw)                               
df_3 = pd.read_excel(path_log)

def clean_data(df):
    df.columns = df.columns.str.strip().str.upper()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip().str.upper().str.replace(r'[^a-zA-Z0-9_ ]', '-', regex=True)
    return df

df_1 = clean_data(df_1)
df_2 = clean_data(df_2)
df_3 = clean_data(df_3)

df_2['DATE'] = pd.to_datetime(df_2['DATE'], dayfirst=True, errors='coerce')
df_2_filtered = df_2[(df_2['STATUS'] != 'VOID') & (df_2['STATUS'].notna())].copy()

df_1_2 = df_1[['ID_REF', 'AREA']].merge(df_2_filtered, left_on='ID_REF', right_on='CODE', how='inner')
df_3_updated = pd.concat([df_1_2, df_3], ignore_index=True)

df_2 = df_2.dropna(subset=['DATE'])
max_date = df_2['DATE'].max()
df_current = df_2[df_2['DATE'] == max_date]
df_past = df_2[df_2['DATE'] < max_date]

df_1_past = pd.merge(df_1, df_past, left_on='ID_REF', right_on='CODE', how='inner')
df_status_a = df_1_past[df_1_past['STATUS'] == 'PENDING'].copy()
df_status_b = df_1_past[df_1_past['STATUS'] == 'ACTIVE'].copy()

df_1_current = pd.merge(df_1, df_current, left_on='ID_REF', right_on='CODE', how='inner')
df_status_current = df_1_current[df_1_current['STATUS'] == 'PENDING'].copy()

def export_styled(df, path, accent_color):
    if df.empty: return
    
    if 'RESOURCE_ID' in df.columns:
        df['RESOURCE_ID'] = pd.to_numeric(df['RESOURCE_ID'], errors='coerce').astype('Int64')
    if 'DATE' in df.columns:
        df['DATE'] = pd.to_datetime(df['DATE']).dt.strftime('%d-%m-%Y')

    writer = pd.ExcelWriter(path, engine='xlsxwriter')
    df.to_excel(writer, index=False, sheet_name='Data')
    wb, ws = writer.book, writer.sheets['Data']

    fmt_header = wb.add_format({'bold': True, 'border': 1, 'bg_color': '#D3D3D3', 'align': 'center'})
    fmt_accent = wb.add_format({'bg_color': accent_color, 'border': 1})
    fmt_std = wb.add_format({'border': 1})

    for i, col in enumerate(df.columns):
        ws.write(0, i, col, fmt_header)
        ws.set_column(i, i, max(len(col), 15))

    col_idx = df.columns.get_loc('STATUS') if 'STATUS' in df.columns else -1
    for r in range(len(df)):
        for c in range(len(df.columns)):
            val = df.iloc[r, c] if pd.notna(df.iloc[r, c]) else ""
            ws.write(r + 1, c, val, fmt_accent if c == col_idx else fmt_std)
    
    writer.close()

df_3_updated['DATE'] = pd.to_datetime(df_3_updated['DATE']).dt.date
with pd.ExcelWriter(path_log, engine='xlsxwriter') as writer:
    df_3_updated.to_excel(writer, index=False)

export_styled(df_status_current, os.path.join(BASE_DIR, "Report_Current_Pending.xlsx"), '#FF0000')
export_styled(df_status_a, os.path.join(BASE_DIR, "Report_Past_Pending.xlsx"), '#FF0000')
export_styled(df_status_b, os.path.join(BASE_DIR, "Report_Past_Active.xlsx"), '#FFFF00')

def automated_email_distribution(df_curr, df_p_a, df_p_b):
    configs = {
        "REGION_A": {"filter": ["AREA_1"], "to": "manager1@example.com", "cc": "admin@example.com"},
        "REGION_B": {"filter": ["AREA_2"], "to": "manager2@example.com", "cc": "admin@example.com"}
    }

    try:
        outlook = win32.GetActiveObject("Outlook.Application")
    except:
        outlook = win32.Dispatch("Outlook.Application")

    for region, cfg in configs.items():
        html_tables = []
        for df, color in [(df_curr, 'red'), (df_p_a, 'red'), (df_p_b, 'yellow')]:
            filtered = df[df['AREA'].isin(cfg['filter'])]
            if not filtered.empty:
                html_tables.append(filtered.to_html(index=False).replace('class="dataframe"', f'style="border:1px solid black; border-collapse:collapse; font-family:Calibri"'))
        
        if not html_tables: continue

        mail = outlook.CreateItem(0)
        mail.To = cfg['to']
        mail.CC = cfg['cc']
        mail.Subject = f"Automated Status Report - {region}"
        mail.HTMLBody = f"<h3>Status Summary for {region}</h3>" + "<br>".join(html_tables)
        mail.Display() 

warnings.filterwarnings("ignore")
hist_file = os.path.join(BASE_DIR, "Performance_History.xlsx")

def build_analytics():
    results = {}
    files = {'Current': "Report_Current_Pending.xlsx", 'History': "Report_Past_Pending.xlsx"}
    
    for key, name in files.items():
        p = os.path.join(BASE_DIR, name)
        if os.path.exists(p):
            tmp_df = pd.read_excel(p)
            results[key] = tmp_df.groupby(['AREA', 'DATE'])['STATUS'].count().reset_index()

    with pd.ExcelWriter(hist_file, engine='openpyxl') as writer:
        for key, data in results.items():
            data.to_excel(writer, sheet_name=key, index=False)
            ws = writer.sheets[key]
            
            chart = BarChart()
            chart.title = f"Trend - {key}"
            d_ref = Reference(ws, min_col=3, min_row=1, max_row=ws.max_row)
            c_ref = Reference(ws, min_col=2, min_row=2, max_row=ws.max_row)
            chart.add_data(d_ref, titles_from_data=True)
            chart.set_categories(c_ref)
            
            line = LineChart()
            line.add_data(d_ref, titles_from_data=True)
            s = line.series[0]
            s.graphicalProperties.line.solidFill = "FF0000"
            
            chart += line
            ws.add_chart(chart, "E2")

if __name__ == "__main__":
    automated_email_distribution(df_status_current, df_status_a, df_status_b)
    build_analytics()

In [ ]:
Riassunto delle Funzionalità del Codice
Questo script rappresenta un ecosistema completo di automazione RPA (Robotic Process Automation) e Data Analysis, strutturato nelle seguenti fasi:

Web Automation (Selenium): Il codice gestisce l'accesso a un portale web protetto (incluso il superamento di eventuali login SSO), naviga tra i menu, applica filtri di ricerca complessi basati su date e categorie e scarica report in formato Excel in una directory temporanea.

Data Cleaning & Integration (Pandas): Una volta ottenuto il dato grezzo, lo script pulisce le stringhe (rimozione caratteri speciali, normalizzazione testo), gestisce i formati data e integra le informazioni unendo il report scaricato con anagrafiche master locali tramite operazioni di merge e concat.

Advanced Reporting (XlsxWriter): Genera report Excel professionali applicando formattazioni condizionali, stili personalizzati alle intestazioni e auto-adattamento delle colonne per migliorare la leggibilità.

Automated Communication (Win32Com): Integra il processo con Microsoft Outlook. Il sistema filtra i dati per area di competenza e invia automaticamente (o prepara per l'invio) email formattate in HTML contenenti tabelle riassuntive stilizzate ai rispettivi responsabili.

Analytics & Visualization (Openpyxl): Archivia i dati storici in un database Excel e genera grafici dinamici (combinazioni di Istogrammi e Linee) per monitorare i trend dei KPI nel tempo.